This Notebook is used to download data from various Python-friendly sources.

# Berkeley Earth Temperature Data
https://berkeleyearth.org/data/
- [Global_TAVG_Gridded_5deg.nc](https://storage.googleapis.com/berkeley-earth-temperature-hr/global/gridded/Global_TAVG_Gridded_5deg.nc)
- [Global_TAVG_Gridded_0p25deg_2020s](https://berkeleyearth.org/high-resolution-data-access-page/)
  - password: highres

In [ ]:
# imports
from typing import Dict, List, Optional, Tuple
import os

import numpy as np
import pandas as pd
import xarray as xr
from matplotlib import pyplot as plt


In [ ]:
# constants
BASE_PATH = os.path.join('..', 'data')
# raw original dataset
if True:
  DATA_PATH = os.path.join(BASE_PATH, 'raw', 'Global_TAVG_Gridded_0p25deg_2020s.nc')
else:
  DATA_PATH = os.path.join(BASE_PATH, 'raw', 'Global_TAVG_Gridded_5deg.nc')
# intermediate datasets
OUT_INTER_PATH1 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_annual-avg.csv')
OUT_INTER_PATH2 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_anual-avg_filtered.csv')
OUT_INTER_PATH3 = os.path.join(BASE_PATH, 'actual', 'temperature', 'temperature_baseline.csv')
# final dataset
OUT_PATH = os.path.join(BASE_PATH, "actual", "temperature", "temperature.csv")

In [ ]:
def filter_nan(dataframe: pd.DataFrame, column: str) -> pd.DataFrame:
  return dataframe[dataframe[column].notnull()].reset_index(drop=True)

In [ ]:
# load dataset
ds = xr.open_dataset(DATA_PATH, engine="netcdf4")
years = np.floor(ds.time.values).astype(int)
annual = (
    ds["temperature"]
    .assign_coords(year=("time", years))
    .groupby("year")
    .mean()
)
df = annual.to_dataframe().reset_index()
df_filtered = filter_nan(df, "temperature")

df_filtered.head()

In [ ]:
if False:
  df.to_csv(OUT_INTER_PATH1, index=True, index_label='id')
  df_filtered.to_csv(OUT_INTER_PATH2, index=True, index_label='id')

In [ ]:
year_start = 2020
year_end = 2025
baseline = annual.sel(
    year=slice(str(year_start), str(year_end))
).mean("year", skipna=True)

df_baseline = filter_nan(baseline.to_dataframe().reset_index(), "temperature")
print(df_baseline.temperature.min())
print(df_baseline.temperature.max())
#df_baseline.to_csv(OUT_INTER_PATH3, index=True, index_label="id")

In [ ]:
# Analyse temperature extremes
df_ext = df_baseline[df_baseline["temperature"] < -6]
print(df_ext.size)
df_ext.head()

In [ ]:
# analyse values for coordinates in Vienna
df_aut = df_baseline[df_baseline["latitude"] > 48]
df_aut = df_aut[df_aut["latitude"] < 48.25]
df_aut = df_aut[df_aut["longitude"] > 16.3]
df_aut = df_aut[df_aut["longitude"] < 16.4]
df_aut.head(n=100)

## Generate Our Dataset

In [ ]:
# transform data into our internal structure
def compute_pain(temperature_value: float, min_temperature: float, max_temperature: float) -> float:
    temp_range = max_temperature - min_temperature
    temp_offset = min_temperature
    return (temperature_value - temp_offset) / temp_range

def normalize_temperature_dataset(dataframe: pd.DataFrame, category: str = "Temperature") -> pd.DataFrame:
    max_temp = dataframe['temperature'].max()
    min_temp = dataframe['temperature'].min()
    data: List[Dict] = []
    for _, row in dataframe.iterrows():
        lat = row['latitude']
        lon = row['longitude']
        temp = row['temperature']
        #value = 0.5 - 0.5 * np.exp(-(temp - temp_offset) / temp_range)
        #if lon < 0 and lat < 0:
        #    value = 1.0
        value = compute_pain(temp, min_temp, max_temp)
        
        data.append({
            'aggrId': None,
            'value': np.round(value, 5),
            'category': category,
            'lat': lat,
            'lng': lon,
        })

    return pd.DataFrame(data)

In [ ]:
df_baseline = pd.read_csv(OUT_INTER_PATH3, index_col='id')
df_temp = normalize_temperature_dataset(df_baseline)
df_temp.to_csv(OUT_PATH, index=True, index_label="id")

In [ ]:
_max_temp = df_baseline['temperature'].max()
_min_temp = df_baseline['temperature'].min()
_temp_range = _max_temp - _min_temp
_temp_offset = _min_temp

print(_max_temp)
print(_min_temp)
print(_temp_range)
print(_temp_offset)

## Visualize the Number of Missing Values
i.e., NaN values

In [ ]:
value_counts = df_filtered["year"].value_counts(sort=False).to_list()
value_start = 1850
plt.bar(x=range(value_start, value_start + len(value_counts)), height=value_counts)

## Visualize the Temperature Values

In [ ]:
dataset = pd.read_csv(OUT_PATH)

In [ ]:
print("min = ", dataset.value.min())
print("max = ", dataset.value.max())

In [ ]:
plt.plot(dataset.value)

In [ ]:
def my_plot(lat: float, lon: float):
  df_full = annual.to_dataframe().reset_index()
  #df_full = df_full[f_lat - step_size < df_full['latitude']]
  df_full = df_full[df_full['latitude'] == lat]
  #df_full = df_full[f_lon - step_size < df_full['longitude']]
  df_full = df_full[df_full['longitude'] == lon]
  df_full = filter_nan(df_full, "temperature")
  plt.plot(df_full.year, df_full.temperature)

In [ ]:
my_plot(47.5, 12.5)

# Sea Level Rise
https://www.star.nesdis.noaa.gov/socd/lsa/SeaLevelRise/LSA_SLR_maps.php
- [Map of sea level rise from TOPEX and Jason-1,-2,-3](https://www.star.nesdis.noaa.gov/socd/lsa/SeaLevelRise/slr/slr_map_ref.nc)


In [ ]:
data_path = os.path.join(BASE_PATH, "raw", "sea-level-rise_trends.nc")
ds = xr.open_dataset(data_path, engine="netcdf4")
df = filter_nan(ds.to_dataframe().reset_index(), "Sea_level_trends")
df

# Emotional Data (Dummy)

In [ ]:
import os
import pandas as pd

In [ ]:
# constants
BASE_PATH = os.path.join('..', 'data')
EMO_DATASET_PATH = os.path.join(BASE_PATH, "dummy", "emotion-values-dummy.csv")
EMO_WORDS_PATH = os.path.join(BASE_PATH, "dummy", "emotion-mvp-words-dummy.csv")

In [ ]:
# load MVP words into dictionary "word_by_country"
df_words = pd.read_csv(EMO_WORDS_PATH)
word_by_country = {
    str(row["iso3"]): str(row["word"])
    for _, row in df_words.iterrows()
    if pd.notna(row["word"]) and str(row["word"]).strip() != ""
}

Target columns:
id,aggrId,value,category,country,word

Available columns:
- words file
iso3,word
- value file
iso3,anger_moral_injury,climate_fear_anxiety_panic,depression_suicidal_ideation,displacement_exile,emotional_pain_unspecified,general_fear_anxiety_panic,general_physical_pain,grief_solastalgia,helplessness_powerlessness,loss_precarity,no_pain,out_of_scope,relationship_social_pain,shame_guilt_self_blame,trauma_overwhelm,uncertainty_instability

In [ ]:
value_columns = \
    "anger_moral_injury,climate_fear_anxiety_panic,depression_suicidal_ideation,displacement_exile,emotional_pain_unspecified,general_fear_anxiety_panic,general_physical_pain,grief_solastalgia,helplessness_powerlessness,loss_precarity,no_pain,out_of_scope,relationship_social_pain,shame_guilt_self_blame,trauma_overwhelm,uncertainty_instability" \
    .split(",")
#for emotion in value_columns:
#  print(f'EMOTIONS["{emotion}"] = {{ }}')

In [ ]:
df = pd.read_csv(EMO_DATASET_PATH)
df.head()

In [ ]:
# transform dataset
df_emo = pd.read_csv(EMO_DATASET_PATH)

# 1) add a column called "aggrId" that contains pd.NA in every row
df_emo["aggrId"] = pd.NA
# 2) sum up all float values of the columns contained in value_columns, divide it by len(value_columns) and store it in a new column called "value"
df_emo["value"] = (df_emo[value_columns].sum(axis=1) / len(value_columns)).round(5)
# 3) add a column called "category" that contains "emotional" in every row (no category used for emotional data)
df_emo["category"] = "emotional"
# 4) add a column called "word" that contains the word associated with the iso3 code in the dictionary word_by_country
df_emo["word"] = df_emo["iso3"].map(word_by_country, na_action=None)    # todo: define na_action?
# 5) rename the column "iso3" to "country"
df_emo = df_emo.rename(columns={"iso3": "country"})
# 6) drop all columns named in value_columns
df_emo = df_emo.drop(columns=[col for col in value_columns if col in df_emo.columns])
# 7) reorder columns
column_order = ["aggrId", "value", "category", "country", "word"]
df_emo = df_emo[column_order]

# 8) Optiona: drop rows without words
df_emo = df_emo[df_emo["word"].notna()]

df_emo.head()

# Socio-economical Data

## Gini Index
- [Source](https://api.worldbank.org/v2/en/indicator/SI.POV.GINI?downloadformat=csv)
    - extract & use API_SI.POV.GINI_DS2_en_csv_v2_499.csv

In [ ]:
import os
import pandas as pd

In [ ]:
# Constants
BASE_PATH = os.path.join('..', 'data')
DATA_PATH = os.path.join(BASE_PATH, 'raw', 'socioeco', 'API_SI.POV.GINI_DS2_en_csv_v2_499.csv')
OUT_PATH = os.path.join(BASE_PATH, 'actual', 'socioeco', 'gini.csv')


In [ ]:
# Years we're interested in
# 2020-2026... ~40% available
# 2010-2026... ~63% available
# 1960-2026... ~64% available (max)
years = [str(year) for year in range(2010, 2026)]

In [ ]:
# The downloaded file is a ZIP containing the CSV.
# pandas can read the ZIP URL directly in many environments.
df_gini = pd.read_csv(DATA_PATH, skiprows=4)    # first four rows are meta data

# Keep only the relevant columns
df_gini = df_gini[
    ["Country Code"] + years
].rename(
    columns={
        "Country Code": "SOVA3",
    }
)

# Convert year columns to numeric.
# Non-numeric values become NaN.
df_gini[years] = df_gini[years].apply(pd.to_numeric, errors="coerce")

# Mean of all available observations from 2015–2025.
# skipna=True means missing years are simply ignored.
df_gini["gini"] = df_gini[years].mean(axis=1, skipna=True)

# Number of actual observations contributing to the mean
df_gini["gini_n"] = df_gini[years].count(axis=1)

# Most recent year for which a Gini value exists
df_gini["gini_latest_year"] = (
    df_gini[years]
    .notna()
    .iloc[:, ::-1]
    .idxmax(axis=1)
    .where(df_gini[years].notna().any(axis=1))
    .astype("Int64")
)

# Keep only countries with at least one observation
#df_gini = df_gini[df_gini["gini_n"] > 0]

# Final dataset
df_gini = df_gini[
    ["SOVA3", "gini", "gini_n", "gini_latest_year"]
]
df_gini.head()

In [ ]:
df_gini.to_csv(OUT_PATH)

### Visualize Distributions

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
show_as_percent = True
counts = df_gini['gini_n'].value_counts(normalize=show_as_percent).sort_index()
counts = counts.reindex(range(len(years)), fill_value=0)

counts.plot(kind='bar', figsize=(12, 4))
plt.xlabel('gini_n')
plt.ylabel('row count')
plt.title('Count of rows for each gini_n value')
plt.xticks(rotation=0)
plt.show()

## GDP
- [Source](https://datacatalog.worldbank.org/search/dataset/0038130/gdp-ranking)
- [Dataset Download](https://datacatalogfiles.worldbank.org/ddh-published/0038130/DR0046440/GDP.csv)
    - values are given in Million $

In [ ]:
import os
import numpy as np
import pandas as pd

In [ ]:
# Constants
BASE_PATH = os.path.join('..', 'data')
DATA_PATH = os.path.join(BASE_PATH, 'raw', 'socioeco', 'GDP.csv')
OUT_PATH = os.path.join(BASE_PATH, 'actual', 'socioeco', 'gdp.csv')

In [ ]:
df_gdp = pd.read_csv(DATA_PATH, skiprows=4)
df_gdp["gdp"] = df_gdp["US dollars"] \
    .str.replace(",", "", regex=False) \
    .apply(pd.to_numeric, errors="coerce") \
    .astype("Int64")
df_gdp = df_gdp.dropna(subset=["rank"])
df_gdp = df_gdp[
    ["SOVA3", "rank", "gdp"]
]
max_gdp = df_gdp["gdp"].max()
log_max_gdp = np.log(max_gdp)
df_gdp["value"] = (log_max_gdp - np.log(df_gdp["gdp"].astype(float))) / log_max_gdp
df_gdp.to_csv(OUT_PATH)
df_gdp.tail()

In [ ]:
# transform to our socioeconomic dataset columns
df_gdp = pd.read_csv(DATA_PATH, skiprows=4)     # first four rows are just metadata
# convert comma-separated gdp strings to integers
df_gdp["gdp"] = df_gdp["US dollars"] \
    .str.replace(",", "", regex=False) \
    .apply(pd.to_numeric, errors="coerce") \
    .astype("Int64")
# drop non-country rows (i.e., unranked entries)
df_gdp = df_gdp.dropna(subset=["rank"])

# compute pain value (logarithmic relative to the max value)
max_gdp = df_gdp["gdp"].max()
log_max_gdp = np.log(max_gdp)
df_gdp["value"] = (log_max_gdp - np.log(df_gdp["gdp"].astype(float))) / log_max_gdp # pain = 1 - log(gdp) / log(maxGdp)

# introduce our custom columns
df_gdp["id"] = np.arange(1, len(df_gdp) + 1)
df_gdp["aggrid"] = pd.NA
df_gdp["category"] = "GDP"
df_gdp["country"] = df_gdp["SOVA3"]     # rename column

# filter for only our required columns
df_gdp = df_gdp[
    ["id", "aggrid", "value", "category", "country"]
]
df_gdp.to_csv(OUT_PATH, index=False)
df_gdp.head()

### WE STILL HAVE TO MANUALLY REMOVE THE END OF THE FILE THAT CONTAINS COMMENTS

# Deforestation (Optional)

In [ ]:
import ee
import geemap
import pandas as pd


In [ ]:
dataset = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')

tree_cover_vis_param = {
    'bands': ['treecover2000'],
    'min': 0,
    'max': 100,
    'palette': ['black', 'green'],
}

tree_loss_vis_param = {
    'bands': ['lossyear'],
    'min': 0,
    'max': 25,
    'palette': ['yellow', 'red'],
}

m = geemap.Map()
m.add_layer(dataset, tree_cover_vis_param, 'tree cover')
m.add_layer(dataset, tree_loss_vis_param, 'tree loss year')
m

In [ ]:
ee.Initialize()

# Load the Hansen dataset
img = ee.Image("UMD/hansen/global_forest_change_2025_v1_13") \
          .select(["treecover2000", "loss"])

# Aggregate from 30 m to ~10 km
coarse = img.reduceResolution(
    reducer=ee.Reducer.mean(),
    maxPixels=65535
).reproject(
    crs="EPSG:6933",
    #crs="EPSG:4326",
    scale=10000
)

# Whole world
world = ee.Geometry.BBox(-180, -90, 180, 90)

# Sample one point per 10 km pixel
fc = coarse.sample(
    region=world,
    scale=10000,
    geometries=True
)

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=fc,
    description="global_hansen_10km",
    fileFormat="CSV"
)
task.start()